## Example 1: Basic hyperparameter optimization study

In [ ]:
import optuna
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Example 1: Basic hyperparameter optimization study
print("Example 1 - Basic hyperparameter optimization study")

try:
    print(f"Optuna version: {optuna.__version__}")
    
    # Create sample dataset for optimization
    X, y = make_classification(n_samples=100, n_features=10, n_informative=5, 
                               n_redundant=2, random_state=42)
    print(f"Dataset created: {X.shape[0]} samples, {X.shape[1]} features")
    
    # Define objective function
    def objective(trial):
        # Suggest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 10, 100)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        
        # Create model with suggested parameters
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42
        )
        
        # Evaluate with cross-validation
        scores = cross_val_score(model, X, y, cv=3)
        return scores.mean()
    
    # Create study and optimize
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=5)  # Small number for testing
    
    print(f"Number of trials: {len(study.trials)}")
    assert len(study.trials) == 5, len(study.trials)
    print(f"Best value: {study.best_value:.4f}")
    print(f"Best parameters: {study.best_params}")
    
    # Show study statistics
    print(f"Study direction: {study.direction}")
    print(f"Study completed successfully")
    
    print("Example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
    raise
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"optuna example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished")

## Example 2: Samplers, pruners, and advanced study configuration

In [ ]:
import optuna
from optuna.samplers import TPESampler, RandomSampler
from optuna.pruners import MedianPruner, HyperbandPruner
import time

# Example 2: Samplers, pruners, and advanced study configuration
print("Example 2 - Samplers, pruners, and advanced study configuration")

try:
    # Test different samplers
    print("Testing different samplers:")
    
    samplers = {
        'Random': RandomSampler(seed=42),
        'TPE': TPESampler(seed=42),
    }
    
    for sampler_name, sampler in samplers.items():
        print(f"  {sampler_name} sampler: {type(sampler).__name__}")
    
    # Test different pruners
    print("\nTesting different pruners:")
    
    pruners = {
        'Median': MedianPruner(),
        'Hyperband': HyperbandPruner()
    }
    
    for pruner_name, pruner in pruners.items():
        print(f"  {pruner_name} pruner: {type(pruner).__name__}")
    
    # Define objective with pruning
    def objective_with_pruning(trial):
        # Simulate iterative training with intermediate values
        for step in range(5):
            # Simulate training step
            intermediate_value = np.random.random() + step * 0.1
            
            # Report intermediate value
            trial.report(intermediate_value, step)
            
            # Check if trial should be pruned
            if trial.should_prune():
                raise optuna.TrialPruned()
            
            time.sleep(0.01)  # Simulate computation
        
        # Return final value
        return intermediate_value
    
    # Create study with TPE sampler and Median pruner
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42),
        pruner=MedianPruner(n_startup_trials=2, n_warmup_steps=2)
    )
    
    print(f"\nRunning optimization with TPE + Median pruning:")
    study.optimize(objective_with_pruning, n_trials=8)
    
    print(f"Total trials: {len(study.trials)}")
    print(f"Completed trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
    print(f"Pruned trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")
    completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    assert completed + pruned == 8, (completed, pruned)
    
    if study.best_trial:
        print(f"Best value: {study.best_value:.4f}")
        print(f"Best trial number: {study.best_trial.number}")
    
    # Test study storage and resumption
    print(f"\nStudy persistence:")
    print(f"Study name: {study.study_name}")
    print(f"Storage backend: {type(study._storage).__name__}")
    
    print("Example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
    raise
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"optuna example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished")

## Example 3: Parameter suggestion types and search spaces

In [ ]:
import optuna
import numpy as np

# Example 3: Parameter suggestion types and search spaces
print("Example 3 - Parameter suggestion types and search spaces")

try:
    print("Testing different parameter suggestion types:")
    
    def objective_all_types(trial):
        # Integer parameters
        batch_size = trial.suggest_int('batch_size', 16, 128, step=16)
        n_layers = trial.suggest_int('n_layers', 1, 5)
        
        # Float parameters  
        learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
        
        # Categorical parameters
        optimizer = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop'])
        activation = trial.suggest_categorical('activation', ['relu', 'tanh', 'sigmoid'])
        
        # Discrete uniform (removed in newer versions, using int with step)
        embedding_dim = trial.suggest_int('embedding_dim', 32, 256, step=32)
        
        print(f"  Trial {trial.number}:")
        print(f"    batch_size: {batch_size}")
        print(f"    n_layers: {n_layers}")
        print(f"    learning_rate: {learning_rate:.6f}")
        print(f"    dropout_rate: {dropout_rate:.3f}")
        print(f"    weight_decay: {weight_decay:.6f}")
        print(f"    optimizer: {optimizer}")
        print(f"    activation: {activation}")
        print(f"    embedding_dim: {embedding_dim}")
        
        # Simulate model performance based on parameters
        score = np.random.random()
        
        # Add some logic to make certain combinations better
        if optimizer == 'adam' and 0.001 <= learning_rate <= 0.01:
            score += 0.1
        if 0.2 <= dropout_rate <= 0.3:
            score += 0.05
        if n_layers == 3:
            score += 0.05
            
        return score
    
    # Create study for comprehensive parameter testing
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
    study.optimize(objective_all_types, n_trials=6)
    
    print(f"\nOptimization results:")
    assert len(study.trials) == 6, len(study.trials)
    print(f"Best value: {study.best_value:.4f}")
    print(f"Best parameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    # Test conditional parameters
    def objective_conditional(trial):
        classifier_type = trial.suggest_categorical('classifier', ['svm', 'rf', 'xgb'])
        
        if classifier_type == 'svm':
            C = trial.suggest_float('svm_C', 0.1, 100, log=True)
            kernel = trial.suggest_categorical('svm_kernel', ['linear', 'rbf'])
            params = {'C': C, 'kernel': kernel}
        elif classifier_type == 'rf':
            n_estimators = trial.suggest_int('rf_n_estimators', 10, 200)
            max_depth = trial.suggest_int('rf_max_depth', 3, 20)
            params = {'n_estimators': n_estimators, 'max_depth': max_depth}
        else:  # xgb
            learning_rate = trial.suggest_float('xgb_lr', 0.01, 0.3)
            max_depth = trial.suggest_int('xgb_depth', 3, 10)
            params = {'learning_rate': learning_rate, 'max_depth': max_depth}
        
        # Simulate performance
        return np.random.random() + (0.1 if classifier_type == 'rf' else 0)
    
    print(f"\nTesting conditional parameters:")
    study_conditional = optuna.create_study(direction='maximize')
    study_conditional.optimize(objective_conditional, n_trials=4)
    
    print(f"Conditional best: {study_conditional.best_value:.4f}")
    print(f"Conditional params: {study_conditional.best_params}")
    
    print("Example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
    raise
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"optuna example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished")

## Example 4: Multi-objective optimization and study analysis

In [ ]:
import optuna
import numpy as np
import joblib

# Example 4: Multi-objective optimization and study analysis
print("Example 4 - Multi-objective optimization and study analysis")

try:
    # Multi-objective optimization
    print("Testing multi-objective optimization:")
    
    def multi_objective(trial):
        # Suggest parameters
        x = trial.suggest_float('x', -10, 10)
        y = trial.suggest_float('y', -10, 10)
        
        # Two conflicting objectives
        objective1 = x**2 + y**2  # Minimize distance from origin
        objective2 = (x - 5)**2 + (y - 5)**2  # Minimize distance from (5, 5)
        
        return objective1, objective2
    
    # Create multi-objective study
    study = optuna.create_study(directions=['minimize', 'minimize'])
    study.optimize(multi_objective, n_trials=10)
    
    print(f"Multi-objective trials: {len(study.trials)}")
    print(f"Best trials (Pareto front): {len(study.best_trials)}")
    assert len(study.best_trials) >= 1, len(study.best_trials)
    
    for i, trial in enumerate(study.best_trials[:3]):  # Show first 3 Pareto optimal
        print(f"  Pareto trial {i+1}: values={trial.values}, params={trial.params}")
    
    # Study analysis and statistics
    print(f"\nStudy analysis:")
    
    # Create single-objective study for analysis
    def simple_objective(trial):
        x = trial.suggest_float('x', -5, 5)
        y = trial.suggest_float('y', -3, 3)
        return (x - 2)**2 + (y + 1)**2
    
    analysis_study = optuna.create_study(direction='minimize')
    analysis_study.optimize(simple_objective, n_trials=12)
    
    print(f"Analysis study trials: {len(analysis_study.trials)}")
    assert len(analysis_study.trials) == 12, len(analysis_study.trials)
    print(f"Best value: {analysis_study.best_value:.4f}")
    print(f"Best params: {analysis_study.best_params}")
    
    # Trial statistics
    completed_trials = [t for t in analysis_study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    values = [t.value for t in completed_trials]
    
    if values:
        print(f"Value statistics:")
        print(f"  Mean: {np.mean(values):.4f}")
        print(f"  Std: {np.std(values):.4f}")
        print(f"  Min: {np.min(values):.4f}")
        print(f"  Max: {np.max(values):.4f}")
    
    # Parameter importance (simplified analysis)
    print(f"\nParameter analysis:")
    param_names = list(analysis_study.best_params.keys())
    print(f"Optimized parameters: {param_names}")
    
    # Trials dataframe-like analysis
    print(f"Trial state distribution:")
    state_counts = {}
    for trial in analysis_study.trials:
        state = trial.state.name
        state_counts[state] = state_counts.get(state, 0) + 1
    
    for state, count in state_counts.items():
        print(f"  {state}: {count}")
    
    # User attributes and system attributes
    print(f"\nTrial attributes:")
    if analysis_study.trials:
        trial = analysis_study.trials[-1]
        print(f"  Trial number: {trial.number}")
        print(f"  Trial state: {trial.state}")
        print(f"  Trial datetime_start: {trial.datetime_start is not None}")
        print(f"  Trial datetime_complete: {trial.datetime_complete is not None}")
        print(f"  Trial duration: {trial.duration is not None}")
    
    print("Example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
    raise
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"optuna example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished")

## Example 5: Advanced features and integration patterns

In [ ]:
import optuna
import numpy as np

# Example 5: Advanced features and integration patterns
print("Example 5 - Advanced features and integration patterns")

try:
    print("Testing advanced Optuna features:")
    
    # Custom sampler example
    print("\nTesting callbacks and logging:")
    
    def objective_with_callbacks(trial):
        # Suggest parameters
        a = trial.suggest_float('a', -5, 5)
        b = trial.suggest_float('b', -5, 5)
        
        # Set user attributes
        trial.set_user_attr('parameter_sum', a + b)
        trial.set_user_attr('parameter_product', a * b)
        
        # Simulate training epochs with intermediate reporting
        for epoch in range(3):
            intermediate_score = (a - 1)**2 + (b + 2)**2 + np.random.normal(0, 0.1)
            trial.report(intermediate_score, epoch)
            
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        final_score = (a - 1)**2 + (b + 2)**2
        return final_score
    
    # Create study with callback
    def print_callback(study, trial):
        if trial.state == optuna.trial.TrialState.COMPLETE:
            print(f"    Trial {trial.number} completed with value {trial.value:.4f}")
    
    study_callbacks = optuna.create_study(
        direction='minimize',
        pruner=MedianPruner(n_startup_trials=1)
    )
    
    study_callbacks.optimize(
        objective_with_callbacks, 
        n_trials=6, 
        callbacks=[print_callback]
    )
    
    print(f"Callback study completed: {len(study_callbacks.trials)} trials")
    print(f"Best value: {study_callbacks.best_value:.4f}")
    
    # Show user attributes
    if study_callbacks.best_trial.user_attrs:
        print(f"Best trial user attributes:")
        for key, value in study_callbacks.best_trial.user_attrs.items():
            print(f"  {key}: {value}")
    
    # Test study enqueue and fixed parameters
    print(f"\nTesting parameter enqueueing:")
    
    def simple_quadratic(trial):
        x = trial.suggest_float('x', -10, 10)
        y = trial.suggest_float('y', -10, 10)
        return x**2 + y**2
    
    enqueue_study = optuna.create_study(direction='minimize')
    
    # Enqueue specific parameter combinations to try first
    enqueue_study.enqueue_trial({'x': 0.0, 'y': 0.0})  # Should be optimal
    enqueue_study.enqueue_trial({'x': 1.0, 'y': 1.0})
    enqueue_study.enqueue_trial({'x': -1.0, 'y': -1.0})
    
    enqueue_study.optimize(simple_quadratic, n_trials=6)
    
    print(f"Enqueued study best value: {enqueue_study.best_value:.4f}")
    print(f"Enqueued study best params: {enqueue_study.best_params}")
    assert enqueue_study.trials[0].params == {'x': 0.0, 'y': 0.0}, enqueue_study.trials[0].params
    assert enqueue_study.best_value == 0.0, enqueue_study.best_value
    
    # Test study memory optimization and trial limit
    print(f"\nTesting optimization termination:")
    
    def early_stopping_objective(trial):
        x = trial.suggest_float('x', -1, 1)
        value = abs(x)
        
        # Early stopping condition in study (would need callback in real use)
        if value < 0.01:  # Very good result
            study.stop()  # This would work with study reference
        
        return value
    
    # Demonstrate different optimization patterns
    optimization_patterns = {
        'maximize': 'maximize',
        'minimize': 'minimize'
    }
    
    for pattern_name, direction in optimization_patterns.items():
        pattern_study = optuna.create_study(direction=direction)
        
        def pattern_objective(trial):
            x = trial.suggest_float('x', -2, 2)
            return x**2 if direction == 'minimize' else -(x**2)
        
        pattern_study.optimize(pattern_objective, n_trials=3)
        print(f"{pattern_name} study best: {pattern_study.best_value:.4f}")
    
    # Integration patterns
    print(f"\nIntegration capabilities:")
    print(f"  Study persistence: In-memory storage")
    print(f"  Distributed optimization: Supported with RDB storage")
    print(f"  Visualization: optuna-dashboard available")
    print(f"  ML framework integration: Available")
    print(f"  Custom samplers/pruners: Supported")
    
    print("Example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
    raise
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"optuna example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished")

## Example 6: optuna database storage and CLI

In [ ]:
import os, shutil, subprocess, sys, tempfile
import optuna

# Example 6: database storage (SQLite) and the `optuna` command-line tool.
# RDBStorage stores the study through SQLAlchemy and creates its schema with alembic migrations.
optuna.logging.set_verbosity(optuna.logging.WARNING)
with tempfile.TemporaryDirectory() as tmp:
    url = f"sqlite:///{os.path.join(tmp, 'optuna.db')}"
    study = optuna.create_study(study_name="demo", storage=url)
    study.optimize(lambda t: t.suggest_float("x", -1, 1) ** 2, n_trials=3)
    storage = optuna.storages.RDBStorage(url)
    reloaded = optuna.load_study(study_name="demo", storage=storage)
    assert len(reloaded.trials) == 3 and reloaded.best_value == study.best_value, reloaded.trials
    # The schema is at the newest alembic revision.
    assert storage.get_current_version() == storage.get_head_version(), storage.get_current_version()
    # The console script sits next to the interpreter, even when that bin directory is not on PATH.
    cli = shutil.which("optuna") or shutil.which("optuna", path=os.path.dirname(sys.executable))
    assert cli, "the `optuna` console script was not found"
    proc = subprocess.run([cli, "studies", "--storage", url], capture_output=True, text=True, timeout=120)
    assert proc.returncode == 0, proc.stderr
    assert "demo" in proc.stdout, proc.stdout
    print(f"Example 6 - SQLite storage: {len(reloaded.trials)} trials reloaded, schema {storage.get_current_version()}")
    print("Example 6 - the optuna CLI lists the study from the database")